# 🚕 Q-Learning en Taxi-v4 — Trabajo en casa

En este ejercicio aplicará Q-Learning al ambiente **Taxi-v4** de Gymnasium.

La lógica es la misma trabajada en FrozenLake:

$$
(s_t,a_t) \rightarrow (r_{t+1},s_{t+1})
$$

y la actualización:

$$
Q(s_t,a_t)
\leftarrow
Q(s_t,a_t)
+
\alpha
\left[
r_{t+1}
+
\gamma \max_a Q(s_{t+1},a)
-
Q(s_t,a_t)
\right]
$$

## Objetivo

Implementar y analizar un agente Q-Learning capaz de aprender a recoger un pasajero y llevarlo a su destino.


## 1. Preparación

Instale Gymnasium si es necesario:

```bash
pip install gymnasium[toy-text]
```


In [ ]:
import gymnasium as gym
import numpy as np
import random
import matplotlib.pyplot as plt

from IPython.display import HTML
from matplotlib import animation


## 2. Crear el ambiente

Taxi tiene un número de estados mucho mayor que FrozenLake.

Cada estado codifica:

- posición del taxi,
- ubicación del pasajero,
- destino del pasajero.

Las acciones posibles son:

| Acción | Significado |
|---|---|
| 0 | South |
| 1 | North |
| 2 | East |
| 3 | West |
| 4 | Pickup |
| 5 | Dropoff |


In [ ]:
env = gym.make("Taxi-v4", render_mode="rgb_array")

print("Número de estados:", env.observation_space.n)
print("Número de acciones:", env.action_space.n)


### Pregunta 1

¿Cuántos estados y cuántas acciones tiene Taxi-v4?

Explique brevemente por qué Taxi tiene muchos más estados que FrozenLake.

R/= Taxi-v4 tiene 500 estados y 6 acciones. Esto se obtiene combinando las 25 posiciones posibles del taxi, las 5 situaciones posibles del pasajero y los 4 destinos: 25 × 5 × 4 = 500.

En FrozenLake básicamente se necesita saber la posición del agente dentro del mapa. En Taxi, además de la posición del taxi, hay que saber dónde está el pasajero y cuál es su destino. Al combinar toda esa información se generan muchas más posibilidades.



## 3. Observar una interacción

Ejecute una acción aleatoria y observe qué devuelve el ambiente.


In [ ]:
state, info = env.reset(seed=42)

action = env.action_space.sample()

next_state, reward, terminated, truncated, info = env.step(action)

print("Estado:", state)
print("Acción:", action)
print("Nuevo estado:", next_state)
print("Recompensa:", reward)
print("Terminated:", terminated)


### Pregunta 2

En la interacción anterior identifique:

$$
s_t,\quad a_t,\quad r_{t+1},\quad s_{t+1}
$$

¿Qué representa cada elemento?

R/= $s_t$ es el estado en el que se encontraba el taxi antes de actuar. $a_t$ corresponde a la acción que se escogió, en este caso mediante la selección disponible en el ambiente. $r_{t+1}$ es la recompensa obtenida después de ejecutar esa acción. Finalmente, $s_{t+1}$ es el nuevo estado al que llega el ambiente.

En conjunto, estos cuatro elementos representan una transición: se parte de un estado, se realiza una acción, se recibe una recompensa y se pasa al siguiente estado. Esa información es la que después utiliza Q-Learning para actualizar sus valores.



## 4. Inicializar la Q-table

Cada fila corresponde a un estado y cada columna a una acción.

Inicialmente:

$$
Q(s,a)=0
$$


In [ ]:
n_states = env.observation_space.n
n_actions = env.action_space.n

Q = np.zeros((n_states, n_actions))

print("Shape de Q:", Q.shape)
Q[:5]


### Pregunta 3

¿Cuántos valores debe aprender el agente en total?

Calcule:

$$
|\mathcal{S}| \times |\mathcal{A}|
$$

R/= Hay que aprender un valor para cada combinación entre estado y acción. Entonces:

$$
500 \times 6 = 3000
$$

Por lo tanto, la tabla Q contiene 3000 valores que el agente va ajustando durante el entrenamiento.



## 5. Política $\epsilon$-greedy

Implemente una función que:

- con probabilidad $\epsilon$ seleccione una acción aleatoria;
- en otro caso seleccione:

$$
\arg\max_a Q(s,a)
$$

### Actividad 1
Complete la función.


In [ ]:
def choose_action(Q, state, epsilon, env):
    # con probabilidad epsilon exploramos con una accion al azar
    if random.uniform(0, 1) < epsilon:
        return env.action_space.sample()
    # si no, explotamos la mejor accion conocida para ese estado
    return int(np.argmax(Q[state]))



## 6. Actualización de Q

La regla de actualización es:

$$
Q(s,a)
\leftarrow
Q(s,a)
+
\alpha
\left[
r+
\gamma\max_{a'}Q(s',a')
-
Q(s,a)
\right]
$$

### Actividad 2
Complete la función.


In [ ]:
def update_q(Q, state, action, reward, next_state, alpha, gamma):
    # mejor valor Q posible desde el siguiente estado
    best_next = np.max(Q[next_state])

    # valor objetivo segun la formula de Q-learning
    target = reward + gamma * best_next

    # error TD: diferencia entre el objetivo y el valor actual
    td_error = target - Q[state, action]

    # actualizamos Q(s,a) moviendolo en direccion del error
    Q[state, action] += alpha * td_error

## 7. Entrenamiento

Ahora implemente el ciclo completo de Q-Learning.

En cada episodio:

1. reiniciar el ambiente;
2. escoger una acción;
3. ejecutar `env.step(action)`;
4. actualizar $Q(s,a)$;
5. mover el agente a `next_state`;
6. terminar cuando el episodio finalice.

Use inicialmente:

```python
alpha = 0.1
gamma = 0.95
epsilon = 0.1
episodes = 5000
```

### Actividad 3
Complete la función.


In [ ]:
def train_q_learning(
    env,
    Q,
    episodes=5000,
    alpha=0.1,
    gamma=0.95,
    epsilon=0.1,
    max_steps=200
):
    rewards = []

    for episode in range(episodes):
        state, _ = env.reset()
        total_reward = 0

        for _ in range(max_steps):

            # escoger accion con la politica epsilon-greedy
            action = choose_action(Q, state, epsilon, env)

            # ejecutar la accion en el ambiente
            next_state, reward, terminated, truncated, _ = env.step(action)

            # actualizar la Q-table con la transicion observada
            update_q(Q, state, action, reward, next_state, alpha, gamma)

            # avanzar al siguiente estado y acumular recompensa
            state = next_state
            total_reward += reward

            # terminar el episodio si corresponde
            if terminated or truncated:
                break

        rewards.append(total_reward)

    return Q, rewards

## 8. Entrenar el agente

Ejecute el entrenamiento una vez haya completado las funciones anteriores.


In [ ]:
Q_initial = np.zeros((n_states, n_actions))

Q_trained, rewards = train_q_learning(
    env,
    Q_initial.copy(),
    episodes=5000,
    alpha=0.1,
    gamma=0.95,
    epsilon=0.3
)


## 9. Curva de aprendizaje

Observe cómo cambia la recompensa durante el entrenamiento.


In [ ]:
window = 100

moving_average = np.convolve(
    rewards,
    np.ones(window) / window,
    mode="valid"
)

plt.figure(figsize=(10, 4))
plt.plot(moving_average)
plt.xlabel("Episodio")
plt.ylabel("Recompensa promedio")
plt.title(f"Taxi-v3 — recompensa promedio ({window} episodios)")
plt.show()


### Pregunta 4

Describa la curva de aprendizaje.

- ¿La recompensa promedio mejora?
- ¿Después de aproximadamente cuántos episodios comienza a estabilizarse?
- ¿El comportamiento observado indica convergencia perfecta o solamente una política razonablemente buena?

R/=
La recompensa promedio mejora bastante durante el entrenamiento. Al comienzo es muy negativa, alrededor de -320, porque el agente todavía está actuando casi al azar y comete muchos errores.

El cambio más fuerte se observa aproximadamente entre los primeros 1000 y 1200 episodios. Después la curva mejora más lentamente y cerca del episodio 1500 empieza a mantenerse en un rango cercano a 0-5, aunque todavía presenta algunas variaciones.

Yo diría que esto muestra una política que ya funciona razonablemente bien, pero no una convergencia perfecta. Todavía existe exploración debido a epsilon=0.1, así que algunas acciones siguen siendo aleatorias incluso cuando el agente ya aprendió bastante.



## 10. Reproducir un episodio

La siguiente función ejecuta una política greedy usando la Q-table aprendida y guarda los frames del episodio.


In [ ]:
def play_episode(env, Q, max_steps=200, seed=None):
    state, _ = env.reset(seed=seed)

    frames = [env.render()]
    total_reward = 0

    for _ in range(max_steps):

        q_values = Q[state]
        max_q = np.max(q_values)

        best_actions = np.flatnonzero(q_values == max_q)
        action = int(np.random.choice(best_actions))

        next_state, reward, terminated, truncated, _ = env.step(action)

        frames.append(env.render())

        total_reward += reward
        state = next_state

        if terminated or truncated:
            break

    return frames, total_reward


def frames_to_video(frames, interval=500):
    fig = plt.figure(figsize=(6, 4))
    plt.axis("off")

    image = plt.imshow(frames[0])

    def update(frame):
        image.set_data(frame)
        return [image]

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=frames,
        interval=interval,
        blit=True,
        repeat=True
    )

    plt.close(fig)
    return HTML(anim.to_jshtml())


## 11. Comparar antes y después

Primero observe un Taxi sin entrenamiento usando una Q-table en cero.


In [ ]:
frames_initial, reward_initial = play_episode(
    env,
    Q_initial,
    max_steps=50,
    seed=7
)

print("Recompensa total sin entrenamiento:", reward_initial)
frames_to_video(frames_initial, interval=500)


Ahora observe el agente entrenado.


In [ ]:
frames_trained, reward_trained = play_episode(
    env,
    Q_trained,
    max_steps=200,
    seed=7
)

print("Recompensa total después del entrenamiento:", reward_trained)
frames_to_video(frames_trained, interval=500)


### Pregunta 5

Compare los dos episodios.

- ¿Qué diferencias observa en el comportamiento del taxi?
- ¿El taxi sin entrenamiento logra completar la tarea?
- ¿El agente entrenado evita acciones innecesarias?
- ¿Qué evidencia visual le permite afirmar que el agente aprendió?

R/=
El taxi que todavía no ha sido entrenado se mueve sin una estrategia clara. Puede chocar con paredes o intentar recoger y dejar al pasajero cuando no corresponde. Normalmente tampoco consigue completar el viaje dentro del límite de pasos.

Después del entrenamiento se nota una diferencia clara: el taxi se dirige hacia el pasajero, hace el pickup cuando corresponde y luego busca el destino evitando movimientos innecesarios.

La principal evidencia visual es que la ruta del agente entrenado es mucho más organizada y logra completar la tarea en menos pasos. También obtiene una recompensa mucho mejor que la del agente sin entrenamiento, lo que indica que la Q-table aprendió una política útil.



## 12. Analizar la política aprendida

Seleccione un estado cualquiera y observe los valores aprendidos para sus seis acciones.


In [ ]:
state = 123

print("Estado:", state)
print("Q-values:", Q_trained[state])
print("Mejor acción:", np.argmax(Q_trained[state]))


### Pregunta 6

Para el estado seleccionado:

1. ¿Cuál es la acción con mayor valor Q?
2. ¿Qué significa que una acción tenga un valor Q mayor que otra?
3. ¿Por qué no podemos interpretar Q(s,a) únicamente como la recompensa inmediata de ejecutar la acción?

R/=
1. La acción con mayor valor Q es la 3, correspondiente a West, con un valor aproximado de 3.949. Es bastante mayor que los demás valores del estado seleccionado.
2. Un Q más alto indica que, desde ese estado, esa acción tiene un mejor retorno esperado considerando también lo que puede pasar después. Por eso no solo se está evaluando el movimiento actual.
3. Q(s,a) no representa únicamente la recompensa inmediata porque también incorpora las recompensas futuras descontadas mediante gamma. Por eso una acción puede tener una recompensa inmediata aceptable y aun así presentar un Q bajo si conduce a estados poco favorables.



## 13. Experimentación

### Pregunta 7

Compare el nuevo entrenamiento con el original.

Explique cómo el cambio del hiperparámetro afectó:

- velocidad de aprendizaje,
- estabilidad,
- recompensa final,
- comportamiento observado.

R/=
En la prueba se aumentó epsilon de 0.1 a 0.3, así que el agente pasó a explorar con más frecuencia.

- **Velocidad:** la curva empieza más abajo y necesita un tiempo parecido para estabilizarse, porque durante más acciones el agente está tomando decisiones aleatorias.
- **Estabilidad:** se observa más variación en la recompensa, algo esperable al aumentar la exploración.
- **Recompensa:** durante el entrenamiento termina siendo menor, aproximadamente entre -10 y -20, frente al rango cercano a 0-5 del entrenamiento original.
- **Comportamiento:** el agente explora más antes de aprovechar lo aprendido. Esto puede ser útil para descubrir estados, aunque durante el entrenamiento se refleja en una recompensa menor.


## Entrega

El notebook debe contener:

1. implementación de `choose_action`;
2. implementación de `update_q`;
3. implementación de `train_q_learning`;
4. curva de aprendizaje;
5. visualización del agente antes y después del entrenamiento;
6. respuestas a las siete preguntas.

No es necesario modificar las funciones auxiliares de visualización.
